In [1]:
import flexlibs
from SIL.LCModel import *
from SIL.LCModel.Core.KernelInterfaces import ITsString, ITsTextProps, ITsStrFactory, ITsPropsFactory, ITsStrBldr, ITsIncStrBldr, ITsPropsBldr, ITsMultiString, FwTextPropType, FwTextPropVar
from SIL.LCModel.Core.Text import TsStringUtils,TsIncStrBldr, TsStrBldr, TsStringComparer
from SIL.LCModel.Core.Cellar import CellarPropertyType, CellarPropertyTypeFilter, GenDate
from SIL.LCModel.Core.WritingSystems import CoreWritingSystemDefinition
from SIL.LCModel.Infrastructure import IFwMetaDataCacheManaged
from System import Guid, String, DateTime, Byte
from docx.shared import Inches
import unicodedata
import docx
# import tkinter as tk
# from tkinter import ttk

flexlibs.FLExInitialize()

In [2]:
project = flexlibs.FLExProject()
project.OpenProject('tkr-lexicon-flex')
# project.OpenProject('IndonesianTest')
project.ProjectName()
cache = project.project
lp = cache.LangProject
lexicon = lp.LexDbOA
sl = cache.ServiceLocator
mdcm = IFwMetaDataCacheManaged(cache.MetaDataCacheAccessor)
ddbf = cache.DomainDataByFlid
wsv = cache.DefaultVernWs
wsa = cache.DefaultAnalWs

entry = next(iter(lexicon.Entries))


In [3]:
analysis_ws_list = project.GetAllAnalysisWSs()

test_tag = None

# Iterate through each analysis writing system
for ws_tag in analysis_ws_list:
    ws_handle = project.WSHandle(ws_tag)
    ws_name = project.WSUIName(ws_tag)
    
    print(ws_handle)
    
    test_handle = ws_handle
    

wsa = test_handle

# print(chosen_wsa_handle)

# chosen_wsa_handle = 999000002

999000004
999000002
999000007


In [5]:
# returns the form, and if it should split on a punctuation
def get_form(analysis, wsv, wsa) -> tuple[str, bool] | None:
    
    class_name = analysis.ClassName
    hvo = analysis.Hvo
    
    try:
        if class_name == "WfiWordform":
            obj = cache.ServiceLocator.GetService(IWfiWordformRepository).GetObject(hvo)
            return (obj.Form.get_String(wsv).Text, False)
            
        elif class_name == "WfiGloss":
            obj = cache.ServiceLocator.GetService(IWfiGlossRepository).GetObject(hvo)
            analysis = obj.Owner
            word_form = analysis.Owner
            wf = cache.ServiceLocator.GetService(IWfiWordformRepository).GetObject(word_form.Hvo)
            return (wf.Form.get_String(wsv).Text, False)
            
        elif class_name == "PunctuationForm":
            obj = cache.ServiceLocator.GetService(IPunctuationFormRepository).GetObject(hvo)
            return (str(obj.Form), True)
            
        elif class_name == "WfiAnalysis":
            obj = cache.ServiceLocator.GetService(IWfiAnalysisRepository).GetObject(hvo)
            if obj.Owner:
                return (obj.Owner.Form.get_String(wsv).Text, False)
            
    except Exception as e:
        print(f"Error with {class_name}: {e}")

In [6]:
def GetAllCustomFields():
    for flid in mdcm.GetFieldIds():
        if mdcm.IsCustom(flid):
            yield(flid, mdcm.GetOwnClsId(flid), mdcm.GetOwnClsName(flid), mdcm.GetFieldType(flid),
            mdcm.GetFieldName(flid), mdcm.GetFieldWs(flid), mdcm.GetFieldListRoot(flid))
    

customFields = GetAllCustomFields()
for field in customFields :
    print(field[0],field[1],field[2],field[3],field[4],field[5],field[6], sep=", ")

5002500, 5002, LexEntry, 13, SemanticDomainsCheck, -1, 00000000-0000-0000-0000-000000000000
5002501, 5002, LexEntry, 16, SummaryPOS, -3, 00000000-0000-0000-0000-000000000000
5016500, 5016, LexSense, 13, EngReversalCheck, -1, 00000000-0000-0000-0000-000000000000
5016501, 5016, LexSense, 16, PublishedPOS, -3, 00000000-0000-0000-0000-000000000000
5016502, 5016, LexSense, 13, ScratchEng, -1, 00000000-0000-0000-0000-000000000000
5035500, 5035, MoForm, 23, Note, -1, 00000000-0000-0000-0000-000000000000


In [11]:
# Use the Text repository directly instead of project.TextsGetAll()
text_repo = sl.GetService(ITextRepository)
notebook_repo = sl.GetService(IRnGenericRecRepository)

count = 0

paragraphs = []

# Get all texts using the repository
count = 0
for text in text_repo.AllInstances():
    count += 1
    if count > 5:
        break
        
    comment = text.Description.get_String(wsa)
    if comment is not None and hasattr(comment, "Text") and comment.Text is not None:
        if comment.Text.lower() == 'y':
            count += 1
            # Get the title
            title = text.Name.get_String(wsv)
            
            # Access the structured content
            sttext = text.ContentsOA
            tags = sttext.TagsOC
            texts = []

            if sttext is not None and hasattr(sttext, "ParagraphsOS"):
                # print(f"Number of paragraphs: {sttext.ParagraphsOS.Count}")
                # Access baseline text from paragraphs
                for para in sttext.ParagraphsOS:
                    # Cast to IStTxtPara to access Contents
                    if hasattr(para, 'Contents'):
                        # It's already IStTxtPara
                        txtpara = para
                    else:
                        # Cast from IStPara to IStTxtPara
                        txtpara = IStTxtPara(para)
                    paragraphs.append(txtpara)
create_docx("testdoc2", paragraphs)
                
        
print(count)
    # create_docx("testdoc", txtpara)

6


In [8]:
def create_docx(doc_name: String, paragraphs):
    doc = docx.Document()
    doc.sections[0].left_margin = Inches(0.5)
    doc.sections[0].right_margin = Inches(0.5)
    for p in paragraphs:
        create_table(p, doc)
        doc.add_page_break()
    
    doc.save(doc_name + ".docx")
    

In [12]:
def create_table(txtpara, doc: docx.Document):
    char_width = 0.12  # average width per character in inches
    min_width = 0.6    # minimum cell width in inches

    # Define the row types you want - add/remove as needed
    row_types = [
        {"label": "Baseline", "merge": False},
        {"label": "Word for Word", "merge": False}, 
        {"label": "Free", "merge": True},
    ]
    
    # print(txtpara.Contents)

    # pass

    for segment in txtpara.SegmentsOS:
        forms = []
        for analysis in segment.AnalysesRS:
            tup = get_form(analysis, wsv, wsa)
            if tup is not None:
                (form, punctuation) = tup
                # if punctuation and forms[0] is not None:
                #     forms[0].append()
                # else:
                forms.append(form)
            if tup is None:
                print(segment)
        
        # Pre-calculate row width
        row_width = 0
        for idx, form in enumerate(forms):
            row_width += len(form) * char_width
        
        print('row width: ' , row_width)
        
        if row_width > 7.5:
            doc.add_paragraph("long")

        # Create table with dynamic number of rows
        # num_rows = 0
        # while row_width > 7.5:
        #     num_rows += len(row_types)
        #     row_width -= 7.5

        table = doc.add_table(rows=num_rows, cols=(len(forms) + 1))
        table.autofit = False

        # Set first column width
        for row_idx in range(num_rows):
            table.rows[row_idx].cells[0].width = Inches(min_width)

        

        # Set widths for word columns and add baseline text
        for idx, form in enumerate(forms, start=1):
            if form is not None:
                width = max(len(form) * char_width, min_width)
                # Set width for all rows in this column
                for row_idx in range(num_rows):
                    table.rows[row_idx].cells[idx].width = Inches(width)
                
                # Add form text to baseline row (first row)
                table.rows[0].cells[idx].text = form

        # Set row labels and handle merging
        for row_idx, row_type in enumerate(row_types):
            table.rows[row_idx].cells[0].text = row_type["label"]
            
            # Merge cells for free translation rows if specified
            if row_type["merge"] and len(forms) > 0:
                base_cell = table.rows[row_idx].cells[1]
                for cell in table.rows[row_idx].cells[2:]:
                    base_cell = base_cell.merge(cell)

        doc.add_page_break()

In [ ]:
project.CloseProject();